# Notebook 6 — Serving the trained model (API)

FastAPI exposes `GET /health`, `POST /predict`, and `GET /model-info`. Notebook 06 uses **`TestClient`**, which runs lifespan startup hooks so models load automatically from **`model/`** and **`artifacts/`** at repo root — same defaults as `app/main.py`.

Production-style run from repo root:

`uvicorn app.main:app --reload --host 127.0.0.1 --port 8000`

Then open the bundled UI (`GET /`) or send JSON to `/predict`.

**Important:** Commands must be run with the repo root as the working directory so `app/main.py` resolves `model/` and `artifacts/` beside the `app/` package.

## `TestClient` smoke test (`/health` + `/predict`)

Starts the FastAPI lifespan once; models load from defaults in `startup`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    h = client.get("/health").json()
    print("health:", json.dumps(h, indent=2))

    payload = {
        "prompt": "Explain overfitting briefly.",
        "response_a": "Overfitting fits noise; model memorises.",
        "response_b": "It is ok.",
        "model_a_name": "DetailBot",
        "model_b_name": "ShortBot",
    }
    r = client.post("/predict", json=payload)
    r.raise_for_status()
    body = r.json()
    print("prediction:", body.get("prediction"))
    print("probs:", body.get("probabilities"))
    print(
        "first insight keys:",
        list((body.get("feature_insights") or {}).keys())[:5],
    )

## Curl template (manual server run)

After launching `uvicorn app.main:app --port 8000` **from repo root**:

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d "{\"prompt\":\"Hello\",\"response_a\":\"...\",\"response_b\":\"...\"}"
```